In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter

# def regress_spatial(R, v):
#     """
#     Equivalent of u = R @ v
#     """
#     u = R @ v
#     norm = np.linalg.norm(u)
#     return u / norm if norm > 0 else u

# def regress_temporal_no_norm(R, u):
#     """
#     Temporal regression: v = R.T @ u
#     No normalization applied (consistent with C++ code)
#     """
#     return R.T @ u

def update_spatial_init(R, u, v):
    """
    Update u ← normalize(R @ v), and return delta_u = ||u_new - u_old||_2

    Parameters:
    - R: np.ndarray of shape (d, t), the residual matrix
    - u: np.ndarray of shape (d,), current estimate of spatial vector
    - v: np.ndarray of shape (t,), current estimate of temporal vector

    Returns:
    - u_new: np.ndarray of shape (d,), updated spatial component
    - delta_u: float, change in u (for convergence check)
    """
    u_old = u.copy()

    # Regress spatial: u ← R @ v
    u_new = R @ v
    norm = np.linalg.norm(u_new)
    if norm > 0:
        u_new /= norm

    # Compute change: delta_u = ||u_new - u_old||_2
    delta_u = np.linalg.norm(u_new - u_old)

    return u_new, delta_u

def update_temporal_init(R, u, v):
    """
    Update v ← normalize(R.T @ u), and return delta_v = ||v_new - v_old||_2

    Parameters:
    - R: np.ndarray of shape (d, t), the residual matrix
    - u: np.ndarray of shape (d,), current spatial vector
    - v: np.ndarray of shape (t,), current temporal vector

    Returns:
    - v_new: np.ndarray of shape (t,), updated temporal component
    - delta_v: float, change in v (for convergence check)
    """
    v_old = v.copy()

    # Regress temporal: v ← R.T @ u
    v_new = R.T @ u
    norm = np.linalg.norm(v_new)
    if norm > 0:
        v_new /= norm

    # Compute change
    delta_v = np.linalg.norm(v_new - v_old)

    return v_new, delta_v

def estimate_noise_mean_filter(u_img, window_size=5):
    """
    Estimate noise using local mean filter and pixel-wise MSE.
    Approximates the C++ function by computing mean-filter residuals.

    Parameters:
    - u_img: 2D numpy array (d1 x d2)
    - window_size: int, size of the local window (default=5)

    Returns:
    - var_hat: float, estimated local noise variance
    """
    d1, d2 = u_img.shape
    pad = window_size // 2

    # Compute local mean via uniform filter
    local_mean = uniform_filter(u_img, size=window_size, mode='reflect')

    # Residual squared error per pixel
    mse_map = (u_img - local_mean) ** 2

    # Flatten, sort, and return 25th percentile (Q1)
    mses = np.sort(mse_map.flatten())
    var_hat = mses[len(mses) // 4]

    return var_hat

def pmd_full_frame(R, max_components=10, consec_failures=3, rank_one_decomposition=None):
    """
    Full-frame Penalized Matrix Decomposition (PMD) implementation.

    Parameters:
    - R: np.ndarray of shape (d, t), the residual matrix (spatial x time)
    - max_components: int, maximum number of rank-1 components to extract
    - consec_failures: int, number of consecutive failed components allowed before stopping
    - rank_one_decomposition: callable, a function to extract (u, v) from R (to be defined)

    Returns:
    - U: np.ndarray of shape (d, num_components), spatial components
    - V: np.ndarray of shape (t, num_components), temporal components
    """

    d, t = R.shape
    U = np.zeros((d, max_components))
    V = np.zeros((t, max_components))

    keep_flag = np.ones(consec_failures, dtype=int)
    good = 0

    for k in range(max_components):
        u, v, success = rank_one_decomposition(R)

        keep_flag[k % consec_failures] = success

        # Stopping criterion: check if recent components are all bad
        if success < 0:
            max_keep_flag = max(keep_flag)
            if max_keep_flag < 0:
                break
            continue

        # Store components
        U[:, good] = u
        V[:, good] = v

        # Update residual
        R = R - np.outer(u, v)

        good += 1

    return U[:, :good], V[:, :good], R